In [ ]:
from selenium import webdriver
from selenium_stealth import stealth
from bs4 import BeautifulSoup
import requests
import pandas as pd
import re
import time
import matplotlib.pyplot as plt
import seaborn as sns
import json
import numpy as np
import undetected_chromedriver as uc
import random
import os
import tempfile

In [ ]:
driver = webdriver.Chrome()

stealth(
    driver,
    languages=["ru-RU", "ru"],
    vendor="Google Inc.",
    platform="Win32",
    webgl_vendor="Intel Inc.",
    renderer="Intel Iris OpenGL Engine",
    fix_hairline=True,
)

driver.get("https://vkusvill.ru/goods/morozhenoe/")
time.sleep(5)

driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
time.sleep(2)

button = driver.find_element(By.CSS_SELECTOR, "button.js-catalog-load-more")
driver.execute_script("arguments[0].click();", button)
time.sleep(3)

scroll_step = 500
pause = 0.3

current = 0
target = driver.execute_script("return document.body.scrollHeight")

while current < target:
    current += scroll_step
    driver.execute_script(f"window.scrollTo(0, {current});")
    time.sleep(pause)
    target = driver.execute_script("return document.body.scrollHeight")

soup = BeautifulSoup(driver.page_source, "html.parser")
driver.quit()

In [ ]:
soup_ = soup.find(
    "div", class_="ProductCards__list js-datalayer-catalog-list js-log-place"
)
names = soup_.find_all(
    "a",
    class_="ProductCard__link rtext _desktop-md _tablet-sm _mobile-xsm js-datalayer-catalog-list-name js-product-detail-link",
)

name = names[-1]["title"].strip()
print(name)

price = soup_.find_all("span", class_="js-datalayer-catalog-list-price hidden")[
    -1
].text.strip()
print(price)

weight = soup_.find_all(
    "div", class_="ProductCard__weight rtext _desktop-sm _mobile-xsm nobr"
)[-1].text.strip()
print(weight)

rate = soup_.find_all(
    "div", class_="ProductCard__ratingText rtext _desktop-sm _mobile-xsm"
)[-1].text.strip()
print(rate)

url = f'https://vkusvill.ru/{soup_.find_all(
    "a",
    class_="ProductCard__link rtext _desktop-md _tablet-sm _mobile-xsm js-datalayer-catalog-list-name js-product-detail-link",
)[-10]["href"]}'
print(url)

In [ ]:
driver = webdriver.Chrome()

stealth(
    driver,
    languages=["ru-RU", "ru"],
    vendor="Google Inc.",
    platform="Win32",
    webgl_vendor="Intel Inc.",
    renderer="Intel Iris OpenGL Engine",
    fix_hairline=True,
)

driver.get(url)
time.sleep(2)


driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")


url_html = BeautifulSoup(driver.page_source, "html.parser")
driver.quit()

In [ ]:
n_reviews = url_html.find(
    "div",
    class_="VV23_DetailProdPageInfoTabs__HeaderTogglerCount js-product-toggler-counter",
).text.strip()
print(n_reviews)

In [ ]:
cards = soup_.select("div.ProductCard")

driver = webdriver.Chrome()
stealth(
    driver,
    languages=["ru-RU", "ru"],
    vendor="Google Inc.",
    platform="Win32",
    webgl_vendor="Intel Inc.",
    renderer="Intel Iris OpenGL Engine",
    fix_hairline=True,
)

data = []

for i, card in enumerate(cards):

    rate_tag = card.find(
        "div", class_="ProductCard__ratingText rtext _desktop-sm _mobile-xsm"
    )
    rate = rate_tag.text.strip() if rate_tag else "Ждет оценку"
    if rate == "Ждет оценку":
        continue

    name_tag = card.find(
        "a",
        class_="ProductCard__link rtext _desktop-md _tablet-sm _mobile-xsm js-datalayer-catalog-list-name js-product-detail-link",
    )
    name = name_tag["title"].strip() if name_tag else None
    url = f"https://vkusvill.ru{name_tag['href']}" if name_tag else None

    price_tag = card.find("span", class_="js-datalayer-catalog-list-price hidden")
    price = price_tag.text.strip() if price_tag else None

    weight_tag = card.find(
        "div", class_="ProductCard__weight rtext _desktop-sm _mobile-xsm nobr"
    )
    weight = weight_tag.text.strip() if weight_tag else None

    n_reviews = None
    try:
        driver.get(url)
        time.sleep(2)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
        time.sleep(1)
        product_soup = BeautifulSoup(driver.page_source, "html.parser")
        n_reviews_tag = product_soup.find(
            "div",
            class_="VV23_DetailProdPageInfoTabs__HeaderTogglerCount js-product-toggler-counter",
        )
        n_reviews = n_reviews_tag.text.strip() if n_reviews_tag else None
    except Exception as e:
        print(f"Ошибка на {url}: {e}")

    data.append(
        {
            "name": name,
            "price": price,
            "weight": weight,
            "rate": rate,
            "n_reviews": n_reviews,
            "url": url,
        }
    )


driver.quit()

df = pd.DataFrame(data)
df.head(2)

In [ ]:
df = df.rename(
    columns={
        "name": "Название",
        "price": "Цена, ₽",
        "weight": "Вес, г",
        "rate": "Рейтинг",
        "n_reviews": "Кол-во отзывов",
        "url": "Ссылка",
    }
)

df["Цена, ₽"] = df["Цена, ₽"].astype(float)
df = df[
    df["Рейтинг"].apply(lambda x: str(x).replace(".", "").replace(",", "").isdigit())
].reset_index(drop=True)
df["Рейтинг"] = df["Рейтинг"].astype(float)
df["Кол-во отзывов"] = df["Кол-во отзывов"].astype(float)


def parse_weight(s):
    s = s.split(" ")
    s = s[0]
    s = float(s)
    return s


df["Вес, г"] = df["Вес, г"].apply(parse_weight)

df["Цена за 100 г, ₽"] = (df["Цена, ₽"] / df["Вес, г"] * 100).round(2)

df.head(2)

In [ ]:
df.to_csv("vkusvill_morozhenoe.csv", index=False)

In [ ]:
with open("names.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(df["Название"].astype(str)))

## Необычные вкусы

In [ ]:
driver = webdriver.Chrome()
url = "https://vkusvill.ru/goods/morozhenoe/neobychnye-vkusy/"
stealth(
    driver,
    languages=["ru-RU", "ru"],
    vendor="Google Inc.",
    platform="Win32",
    webgl_vendor="Intel Inc.",
    renderer="Intel Iris OpenGL Engine",
    fix_hairline=True,
)

driver.get(url)
time.sleep(2)


driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")


nv_html = BeautifulSoup(driver.page_source, "html.parser")
driver.quit()
unusual_flavors = [
    i["title"]
    for i in nv_html.find_all(
        "a",
        class_="ProductCard__link rtext _desktop-md _tablet-sm _mobile-xsm js-datalayer-catalog-list-name js-product-detail-link",
    )
]

with open("unusual_flavors.json", "w", encoding="utf-8") as f:
    json.dump(unusual_flavors, f, ensure_ascii=False)

print(f"Сохранено {len(unusual_flavors)} необычных вкусов в unusual_flavors.json")

In [ ]:
CSV_PATH = "vkusvill_morozhenoe.csv"

HIERARCHY = {
    "ПП": 1,
    "Необычный вкус": 1,
    "Эскимо": 2,
    "Рожок": 2,
    "Вафельный стаканчик": 2,
    "Большая порция": 2,
    "Сорбет/лёд": 3,
    "Пломбир": 3,
    "Стаканчик (не вафельный)": 4,
}


def _normalize(s):
    if not isinstance(s, str):
        return ""
    s = s.replace("\xa0", " ").replace("&nbsp;", " ")
    s = re.sub(r'["«»""\'\'.,()]', "", s)
    s = s.replace("—", "-").replace("–", "-")
    s = re.sub(r"\s+", " ", s)
    return s.strip().lower()


def _parse_weight(s):
    if isinstance(s, (int, float)):
        return float(s) if pd.notna(s) else np.nan
    m = re.search(r"[\d.,]+", str(s))
    return float(m.group().replace(",", ".")) if m else np.nan


def build_df(csv_path: str = CSV_PATH) -> pd.DataFrame:
    """Сборка датафрейма из CSV."""
    df = pd.read_csv(csv_path)

    df["Цена, ₽"] = df["Цена, ₽"].apply(
        lambda x: (
            float(re.search(r"[\d.,]+", str(x)).group().replace(",", "."))
            if re.search(r"[\d.,]+", str(x))
            else np.nan
        )
    )
    df = df[
        df["Рейтинг"].apply(
            lambda x: str(x).replace(".", "").replace(",", "").isdigit()
        )
    ].reset_index(drop=True)
    df["Рейтинг"] = df["Рейтинг"].astype(str).str.replace(",", ".").astype(float)
    df["Кол-во отзывов"] = pd.to_numeric(df["Кол-во отзывов"], errors="coerce").fillna(
        0
    )
    df["Вес, г"] = df["Вес, г"].apply(_parse_weight)

    df["Цена за 100 г, ₽"] = (df["Цена, ₽"] / df["Вес, г"] * 100).round(2)

    df = df.drop_duplicates(subset=["Название"]).reset_index(drop=True)

    try:
        with open("unusual_flavors.json", encoding="utf-8") as f:
            unusual = json.load(f)
        unusual_set = {_normalize(n) for n in unusual}
    except FileNotFoundError:
        unusual_set = set()
        print(
            "ВНИМАНИЕ: unusual_flavors.json не найден — 'Необычный вкус' = False везде"
        )
    df["Необычный вкус"] = df["Название"].apply(lambda x: _normalize(x) in unusual_set)

    def has(pattern):
        return df["Название"].str.contains(pattern, case=False, regex=True, na=False)

    df["ПП"] = has(r"без\s*доб|без\s*сахар|протеин|безлакт|без\s*лакт")
    df["Вафельный стаканчик"] = has(r"вафельн\w*\s+стакан|ваф\.?\s*стак")
    df["Рожок"] = has(r"рожк|рожок|в\s*рожке")
    df["Сорбет/лёд"] = has(r"сорбет|^лёд|^лед\b|фруктово-ягодн")
    df["Эскимо"] = has(r"эскимо")
    df["Пломбир"] = has(r"пломбир")
    df["Большая порция"] = df["Вес, г"] >= 250
    other_form = has(r"сэндвич|брикет|моти|торт")
    df["Стаканчик (не вафельный)"] = ~(
        df["Вафельный стаканчик"]
        | df["Рожок"]
        | df["Эскимо"]
        | df["Большая порция"]
        | other_form
    )

    df = df[df["Рейтинг"] >= 4.5].reset_index(drop=True)

    def get_top_niches(row, max_niches=2):
        active = [(c, lvl) for c, lvl in HIERARCHY.items() if row[c]]
        active.sort(key=lambda x: x[1])
        return [c for c, _ in active[:max_niches]]

    df["Топ-ниши"] = df.apply(get_top_niches, axis=1)

    medians = {
        niche: {
            "price": df.loc[df[niche], "Цена за 100 г, ₽"].median(),
            "reviews": df.loc[df[niche], "Кол-во отзывов"].median(),
        }
        for niche in HIERARCHY
    }

    def components(row):
        niches = row["Топ-ниши"]
        if not niches:
            return pd.Series(
                {
                    "score": np.nan,
                    "Премия по цене": np.nan,
                    "Относит. популярность": np.nan,
                    "Главная ниша": None,
                    "Вторая ниша": None,
                }
            )

        def nc(niche):
            m = medians[niche]
            rp = row["Цена за 100 г, ₽"] / m["price"]
            pop = np.log1p(row["Кол-во отзывов"]) / np.log1p(m["reviews"])
            return rp, pop

        if len(niches) == 1:
            rp, pop = nc(niches[0])
            return pd.Series(
                {
                    "score": rp * pop,
                    "Премия по цене": rp,
                    "Относит. популярность": pop,
                    "Главная ниша": niches[0],
                    "Вторая ниша": None,
                }
            )
        p1, r1 = nc(niches[0])
        p2, r2 = nc(niches[1])
        return pd.Series(
            {
                "score": 0.9 * p1 * r1 + 0.1 * p2 * r2,
                "Премия по цене": 0.9 * p1 + 0.1 * p2,
                "Относит. популярность": 0.9 * r1 + 0.1 * r2,
                "Главная ниша": niches[0],
                "Вторая ниша": niches[1],
            }
        )

    comp = df.apply(components, axis=1)
    df[comp.columns] = comp

    df = df.sort_values("score", ascending=False).reset_index(drop=True)
    df.attrs["medians"] = medians
    return df


df = build_df()
df.head(2)

## Разбиение по нишам

Иерархия ниш (уровень = приоритет при выборе главной ниши):

- Уровень 1: ПП, Необычный вкус
- Уровень 2: Эскимо, Рожок, Вафельный стаканчик, Большая порция
- Уровень 3: Сорбет/лёд, Пломбир
- Уровень 4: Стаканчик (не вафельный)

Скоринг:

1. Для товара находим все ниши, в которые он попадает
2. Сортируем по уровню (1-4)
3. Берём топ-2 ниши с высшего уровня
4. score = 0.9 * score_ниша_1 + 0.1 * score_ниша_2

Ценовая премия:

$$\text{Премия} = \frac{\text{цена за 100 г}}{\text{медиана цены за 100 г в нише}}$$

Относительная популярность:

$$\text{Популярность} = \frac{\ln(1 + \text{кол-во отзывов})}{\ln(1 + \text{медиана кол-ва отзывов в нише})}$$

$$\text{Score} = \text{Премия} \times \text{Популярность}$$

Рейтинг в формулу не входит, используется как фильтр (>= 4.5).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

top25 = df.sort_values("score", ascending=False).head(25).reset_index(drop=True)
palette = sns.color_palette("husl", n_colors=len(top25))

fig, ax = plt.subplots(figsize=(11, 9))

for i, row in top25.iterrows():
    ax.scatter(
        row["Относит. популярность"],
        row["Премия по цене"],
        s=200,
        color=palette[i],
        edgecolor="white",
        linewidth=1.5,
        label=f"{i+1}. {row['Название'][:50]} (score {row['score']:.2f})",
        zorder=3,
    )
    ax.annotate(
        str(i + 1),
        (row["Относит. популярность"], row["Премия по цене"]),
        xytext=(7, 7),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold",
        color="#333333",
        zorder=4,
    )

ax.axhline(1, color="black", linestyle="--", alpha=0.4, linewidth=1)
ax.axvline(1, color="black", linestyle="--", alpha=0.4, linewidth=1)

ax.set_xlabel("Относительная популярность (>1 = популярнее медианы ниши)", fontsize=11)
ax.set_ylabel("Премия по цене (>1 = дороже медианы ниши)", fontsize=11)
ax.set_title("Топ-25 кандидатов", fontsize=14, fontweight="bold", pad=15)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.10),
    ncol=2,
    fontsize=9,
    frameon=False,
)

sns.despine()
plt.tight_layout()
plt.show()